### Config Class

In the BaseSettings I fixed: 

- delete duplicate WEATHER_API_BASE_URL: str ="" that overwrote the first URL
- delete TEMP_MIN instead of TEMP_MIN = str = "Needs Debugging"

### Helper Functions

classify_temperature(): 
- fixed condition for minimal temperature
- Protect against cases where DataType is unknown or invalid strings
- Converted temp_celsius to float to accept strings containing values

get_weather_description():
- Protect against cases where weather_code could come from API as a string

parse_utc_offset():
- Protect against whitespace, empty strings, None values, etc
- Protect against unlogical values like +99:99
- Protect against API fetches with wrong format

seconds_to_ufc_offset():
- Create function to transform format from ufc_offset obtained in API to expected format in the other functions

### Graph Builder

- Add the missing edged that fetches the weather data and sends it to `generate_weather_info`": `builder.add_edge("fetch_location_data", "fetch_weather_data")` + `builder.add_edge("fetch_weather_data", "generate_weather_info")`
- Correrctly compile the graph with `builderr.compile()`

### Nodes

fetch_location_data():
- Fixed naming of required fields to match those in the API anndn in t he LocationData schema
- API returns offset in seconds, fix to transform to expected format in `parse_utc_offset()`

generate_weather_info():
- Uncomment wind_unit to later call it in the f-string format
- Move the fetching of location, weather, and units inside the `try` block

### main 

- Fix incorrect guard: `if __name__ = "__main__"` 
- Correct the object type of state to dictionary since the `WeatherAgentState` is a `TypedDict`.

# Implementation of Weather Agent



### Imports

In [1]:
from weather_agent.graph import weather_agent
from weather_agent.components.config import config
from weather_agent.components.state import WeatherAgentState
from weather_agent.components.helper_functions import (
    classify_temperature,
    get_weather_description,
    get_greeting,
    seconds_to_utc_offset,
    parse_utc_offset,
    format_local_time,
)

### Test Different Functions and Cases

#### Test helper function: `classify_temperature`

In [10]:
temperature_tests = [
    (-5, "cold"),
    (0, "cold"),
    (12, "cool"),
    (20, "comfortable"),
    (28, "warm"),
    (35, "hot"),
    (None, "unknown"),  # only passes if function protects against None
    ("25", "comfortable"),  # only passes if function converts strings to float
    ("invalid", "unknown"),
]

for temp, expected in temperature_tests:
    try:
        actual = classify_temperature(temp)
    except Exception as e:
        actual = f"ERROR: {type(e).__name__}: {e}"

    print(f"classify_temperature({temp!r}) -> {actual!r} | expected: {expected!r}")

classify_temperature(-5) -> 'cold' | expected: 'cold'
classify_temperature(0) -> 'cold' | expected: 'cold'
classify_temperature(12) -> 'cool' | expected: 'cool'
classify_temperature(20) -> 'comfortable' | expected: 'comfortable'
classify_temperature(28) -> 'warm' | expected: 'warm'
classify_temperature(35) -> 'hot' | expected: 'hot'
classify_temperature(None) -> 'unknown' | expected: 'unknown'
classify_temperature('25') -> 'comfortable' | expected: 'comfortable'
classify_temperature('invalid') -> 'unknown' | expected: 'unknown'


#### Test helper function: `get_weather_description`

Check the known WMO weather codes and unknown values

In [11]:
weather_code_tests = [0, 1, 2, 3, 61, 95, 999, "61", None]

for code in weather_code_tests:
    try:
        result = get_weather_description(code)
    except Exception as e:
        result = f"ERROR: {type(e).__name__}: {e}"

    print(f"get_weather_description({code!r}) -> {result!r}")

get_weather_description(0) -> 'Clear sky'
get_weather_description(1) -> 'Mainly clear'
get_weather_description(2) -> 'Partly cloudy'
get_weather_description(3) -> 'Overcast'
get_weather_description(61) -> 'Slight rain'
get_weather_description(95) -> 'Thunderstorm'
get_weather_description(999) -> 'Weather code 999'
get_weather_description('61') -> 'Slight rain'
get_weather_description(None) -> 'Unknown weather condition'


#### Test helper function: `parse_utc_offset()`

Check whether UTC offset strings are converted into `timedelta` objects correctly

In [15]:
utc_offset_tests = [
    "+02:00",
    "-08:00",
    "+05:30",
    "+0530",
    "+2",
    "",
    None,
    "invalid",
]

for offset in utc_offset_tests:
    try:
        result = parse_utc_offset(offset)
    except Exception as e:
        result = f"ERROR: {type(e).__name__}: {e}"

    print(f"parse_utc_offset({offset!r}) -> {result}")

parse_utc_offset('+02:00') -> 2:00:00
parse_utc_offset('-08:00') -> -1 day, 16:00:00
parse_utc_offset('+05:30') -> 5:30:00
parse_utc_offset('+0530') -> 5:30:00
parse_utc_offset('+2') -> 2:00:00
parse_utc_offset('') -> 0:00:00
parse_utc_offset(None) -> 0:00:00
parse_utc_offset('invalid') -> 0:00:00


#### Test helper function: `seconds_to_utc_offset`


In [17]:
offset_second_tests = [7200, -28800, 19800, 0, None, "invalid"]

for seconds in offset_second_tests:
    try:
        result = seconds_to_utc_offset(seconds)
    except Exception as e:
        result = f"ERROR: {type(e).__name__}: {e}"

    print(f"seconds_to_utc_offset({seconds!r}) -> {result!r}")

seconds_to_utc_offset(7200) -> '+02:00'
seconds_to_utc_offset(-28800) -> '-08:00'
seconds_to_utc_offset(19800) -> '+05:30'
seconds_to_utc_offset(0) -> '+00:00'
seconds_to_utc_offset(None) -> '+00:00'
seconds_to_utc_offset('invalid') -> '+00:00'


#### Test helper function: `format_local_time`

Check whether UTC timem strings are converted to local display strings

In [21]:
time_tests = [
    ("2026-05-01T12:00:00Z", "+02:00"),
    ("2026-05-01T12:00:00+00:00", "+05:30"),
    ("2026-05-01T12:00:00", "+02:00"),
    (None, "+02:00"),
    ("invalid", "+02:00"),
]

for utc_time, offset in time_tests:
    try:
        result = format_local_time(utc_time, offset)
    except Exception as e:
        result = f"ERROR: {type(e).__name__}: {e}"

    print(f"format_local_time({utc_time!r}, {offset!r}) -> {result!r}")

format_local_time('2026-05-01T12:00:00Z', '+02:00') -> '12:00 UTC | 14:00 (UTC+02:00)'
format_local_time('2026-05-01T12:00:00+00:00', '+05:30') -> '12:00 UTC | 17:30 (UTC+05:30)'
format_local_time('2026-05-01T12:00:00', '+02:00') -> '12:00 UTC | 14:00 (UTC+02:00)'
format_local_time(None, '+02:00') -> 'Time unavailable'
format_local_time('invalid', '+02:00') -> 'Time unavailable'


#### Test helper function: `generate_weather_info`

Test formatting logic without calling APIs

In [23]:
from components.nodes import generate_weather_info

mock_state: WeatherAgentState = {
    "name": "Daniel",
    "location_data": {
        "city": "Munich",
        "region": "Bavaria",
        "country_name": "Germany",
        "latitude": 48.114,
        "longitude": 11.5422,
        "timezone": "Europe/Berlin",
        "utc_offset": "+02:00",
    },
    "weather_data": {
        "current_weather": {
            "time": "2026-05-01T12:00:00Z",
            "temperature": 21.5,
            "windspeed": 12.3,
            "winddirection": 180,
            "is_day": 1,
            "weathercode": 2,
        },
        "current_weather_units": {
            "temperature": "°C",
            "windspeed": "km/h",
        },
    },
    "weather_info": None,
}

try:
    result_state = generate_weather_info(mock_state.copy())
    print(result_state["weather_info"])
except Exception as e:
    print("ERROR:", type(e).__name__, e)

Time: 12:00 UTC | 14:00 (UTC+02:00)

Good morning, Daniel!

Your current location: Munich, Bavaria, Germany

Current weather conditions:
• Partly cloudy
• Temperature: 21.5°C (comfortable)
• Wind: 12.3km/h


#### Check Failure Scenario 

Verify if `fetch_weather_data` fails when location data is missing

In [24]:
from components.nodes import fetch_weather_data

missing_location_state: WeatherAgentState = {
    "name": "Daniel",
    "location_data": None,
    "weather_data": None,
    "weather_info": None,
}

try:
    fetch_weather_data(missing_location_state)
except Exception as e:
    print("Expected failure:")
    print(type(e).__name__, e)

Expected failure:
Exception Location data not available for weather fetch


### Test Full LangGraph Agent

See if the API is correcly compiled

In [3]:
initial_state: WeatherAgentState = {
    "name": "Daniel",
    "location_data": None,
    "weather_data": None,
    "weather_info": None,
}

try:
    final_state = weather_agent.invoke(initial_state)

    print("=" * 60)
    print("FINAL WEATHER INFO")
    print("=" * 60)
    print(final_state.get("weather_info", "No weather_info returned."))

    print("\nFull final state:")
    print(final_state)

except Exception as e:
    print("Agent failed:")
    print(type(e).__name__, e)

FINAL WEATHER INFO
Time: 15:00 UTC | 17:00 (UTC+02:00)

Good morning, Daniel!

Your current location: Munich, Bavaria, Germany

Current weather conditions:
• Clear sky
• Temperature: 19.9°C (comfortable)
• Wind: 14.4km/h

Full final state:
{'name': 'Daniel', 'location_data': {'city': 'Munich', 'region': 'Bavaria', 'country_name': 'Germany', 'latitude': 48.114, 'longitude': 11.5422, 'timezone': 'Europe/Berlin', 'utc_offset': '+02:00'}, 'weather_data': {'latitude': 48.12, 'longitude': 11.539999, 'generationtime_ms': 0.10323524475097656, 'utc_offset_seconds': 0, 'timezone': 'GMT', 'timezone_abbreviation': 'GMT', 'elevation': 528.0, 'current_weather_units': {'time': 'iso8601', 'interval': 'seconds', 'temperature': '°C', 'windspeed': 'km/h', 'winddirection': '°', 'is_day': '', 'weathercode': 'wmo code'}, 'current_weather': {'time': '2026-05-01T15:00', 'interval': 900, 'temperature': 19.9, 'windspeed': 14.4, 'winddirection': 86, 'is_day': 1, 'weathercode': 0}}, 'weather_info': 'Time: 15:00 U

In [3]:
from weather_agent.components.config import config

print(config.LOCATION_API_URL)

http://ip-api.com/json/


In [4]:
from weather_agent.main import main

main()


WEATHER INFORMATION
Time: 15:45 UTC | 17:45 (UTC+02:00)

Good morning, Juan!

Your current location: Munich, Bavaria, Germany

Current weather conditions:
• Clear sky
• Temperature: 19.4°C (comfortable)
• Wind: 13.7km/h
